อันนี้คือหนูใช้ข้อมูลของ ปี 65-68 โดยต้องมีครบทั้ง 4 ปี และมีการดูในส่วนของรายละเอียดเพิ่มด้วยแล้วค่ะ แต่ว่า พอดูในส่วนของรายละเอียด มันก็จะมีการที่บางทีบางมหาลัยก็ใส่รายละเอียดของคณะ/สาขานั้นๆ แต่บางปีก็ไม่ได้ใส่ เลยทำให้พอเพิ่มเงื่อนไขที่ว่ารายละเอียดต้องตรงกัน ข้อมูลคณะที่ทายก็เลยหายไปเพิ่มไปอีกค่ะTT

In [ ]:
!pip install openpyxl -q

import os, ast, glob
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

os.makedirs('/content/data', exist_ok=True)

MAX_NEW = 100
WINDOW  = 2
ALL_YEARS = [65, 66, 67, 68]

print("Setup complete")

Setup complete


In [ ]:
all_data = []

for file in sorted(glob.glob('/content/data/TCAS*.xlsx')):
    year = int(os.path.basename(file)[4:6])
    if year < 65:
        continue

    df = pd.read_excel(file)
    df['year'] = year

    if year == 65:
        df = df.rename(columns={
            'program_id':                               'รหัสหลักสูตร',
            'program_lookup_programs.faculty_name_th':  'คณะ',
            'university_name':                          'สถาบัน',
            'project_name_th':                          'รายละเอียด',
        })
        df['คะแนนต่ำสุด'] = df['คะแนนต่ำสุด หลังประมวลผลรอบ 2'].fillna(df['คะแนนต่ำสุด'])
        df['คะแนนสูงสุด'] = df['คะแนนสูงสุด หลังประมวลผลรอบ 2'].fillna(df['คะแนนสูงสุด'])

    elif year == 66:
        df = df.rename(columns={'คณะ/สำนักวิชา': 'คณะ'})

    elif year == 67:
        df['คะแนนต่ำสุด'] = df['คะแนนต่ำสุด หลังประมวลผลรอบ 2'].fillna(df['คะแนนต่ำสุด'])
        df['คะแนนสูงสุด'] = df['คะแนนสูงสุด หลังประมวลผลรอบ 2'].fillna(df['คะแนนสูงสุด'])

    elif year == 68:
        df['คะแนนต่ำสุด'] = df['คะแนนต่ำสุด ประมวลผลครั้งที่ 2'].fillna(df['คะแนนต่ำสุด ประมวลผลครั้งที่ 1'])
        df['คะแนนสูงสุด'] = df['คะแนนสูงสุด ประมวลผลครั้งที่ 2'].fillna(df['คะแนนสูงสุด ประมวลผลครั้งที่ 1'])

    scale = MAX_NEW
    df['min_pct'] = df['คะแนนต่ำสุด'] / scale * 100
    df['max_pct'] = df['คะแนนสูงสุด'] / scale * 100

    faculty_col = next((c for c in ['คณะ', 'คณะ/สำนักวิชา'] if c in df.columns), None)
    inst_col    = next((c for c in ['สถาบัน', 'university_name'] if c in df.columns), None)

    all_data.append(pd.DataFrame({
        'program_id':  df['รหัสหลักสูตร'],
        'faculty':     df[faculty_col] if faculty_col else np.nan,
        'institution': df[inst_col]    if inst_col    else np.nan,
        'min_pct':     df['min_pct'],
        'max_pct':     df['max_pct'],
        'criteria':    df['รายละเอียด'] if 'รายละเอียด' in df.columns else np.nan,
        'applied':     df['สมัคร'] if 'สมัคร' in df.columns else np.nan,
        'accepted':    df['รับ']   if 'รับ'   in df.columns else np.nan,
        'year':        year,
    }))
    print(f'ปี {year}: {len(all_data[-1])} แถว')

df_main = pd.concat(all_data, ignore_index=True)
df_main = df_main[(df_main['min_pct'] > 0)].dropna(subset=['min_pct', 'program_id'])
df_main['criteria'] = df_main['criteria'].replace('0', np.nan)
print(f'\nรวม (ปี 65–68): {len(df_main)} แถว')


ปี 65: 4894 แถว
ปี 66: 4670 แถว
ปี 67: 4718 แถว
ปี 68: 4945 แถว

รวม (ปี 65–68): 16057 แถว


In [ ]:
df_main

,program_id,faculty,institution,min_pct,max_pct,criteria,applied,accepted,year
0,10010121300001A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,51.0196,85.4686,คณะวิศวกรรมศาสตร์ สาขาวิชาวิศวกรรมศาสตร์,710.0,465.0,65
1,10010121300501A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,69.6333,80.0961,NaN,306.0,50.0,65
2,10010121301101A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,45.3804,62.6137,NaN,97.0,25.0,65
3,10010121302101A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,61.2921,74.1294,NaN,67.0,10.0,65
4,10010121302501A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,45.2471,53.8137,NaN,144.0,55.0,65
...,...,...,...,...,...,...,...,...,...
19206,49180104901101A,คณะนิเทศศาสตร์,สถาบันการจัดการปัญญาภิวัฒน์,35.0888,47.2360,NaN,3.0,10.0,68
19207,49180104901102A,คณะนิเทศศาสตร์,สถาบันการจัดการปัญญาภิวัฒน์,37.7222,61.2638,NaN,6.0,10.0,68
19209,49180106610801A,คณะการจัดการธุรกิจอาหาร,สถาบันการจัดการปัญญาภิวัฒน์,98.7500,98.7500,NaN,5.0,70.0,68
19215,49180211111701A,คณะพยาบาลศาสตร์,สถาบันการจัดการปัญญาภิวัฒน์,47.7207,57.9040,โครงการ PIM Silver Nurses-Partial Scholarship ...,167.0,15.0,68


In [ ]:
df_main[60:120]

,program_id,faculty,institution,min_pct,max_pct,criteria,applied,accepted,year
64,10010124903201A,คณะรัฐศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,75.0667,87.2196,สาขาวิชาความสัมพันธ์ระหว่างประเทศ เลือกสอบวิชา...,244.0,66.0,65
65,10010124903201A,คณะรัฐศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,75.1167,91.5118,สาขาวิชาความสัมพันธ์ระหว่างประเทศ เลือกสอบวิชา...,258.0,66.0,65
66,10010124903201A,คณะรัฐศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,75.0167,93.7833,สาขาวิชาความสัมพันธ์ระหว่างประเทศ เลือกสอบวิชา...,312.0,66.0,65
67,10010124903201A,คณะรัฐศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,77.0196,78.2167,สาขาวิชาความสัมพันธ์ระหว่างประเทศ เลือกสอบวิชา...,438.0,66.0,65
68,10010125400201A,คณะสถาปัตยกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,41.6667,59.1167,NaN,170.0,22.0,65
69,10010125400301A,คณะสถาปัตยกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,43.0000,68.5000,NaN,177.0,25.0,65
70,10010125400401A,คณะสถาปัตยกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,39.2000,55.0333,NaN,154.0,25.0,65
71,10010125400501A,คณะสถาปัตยกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,55.5000,72.9167,NaN,236.0,10.0,65
72,10010125400701A,คณะสถาปัตยกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,50.2000,59.0667,NaN,193.0,11.0,65
73,10010126610501A,คณะพาณิชยศาสตร์และการบัญชี,จุฬาลงกรณ์มหาวิทยาลัย,63.9824,71.7000,หลักสูตรบัญชีบัณฑิต/หลักสูตรบริหารธุรกิจบัณฑิต...,39.0,10.0,65


In [ ]:
df_main.describe()

,min_pct,max_pct,applied,accepted,year
count,16057.000000,16057.000000,16057.000000,16057.000000,16057.000000
mean,50.201328,65.543688,246.861431,35.297378,66.516410
std,19.076566,19.386074,483.546956,35.810542,1.121703
min,1.308000,2.900000,1.000000,1.000000,65.000000
25%,36.704900,51.916200,25.000000,13.000000,66.000000
50%,50.981300,63.115200,86.000000,28.000000,67.000000
75%,62.250000,80.401300,261.000000,45.000000,68.000000
max,100.000000,100.000000,9244.000000,600.000000,68.000000


In [ ]:
df_main.groupby('year').agg({'faculty':'nunique', 'program_id':'nunique','min_pct':'mean','max_pct':'mean'})

,faculty,program_id,min_pct,max_pct
year,,,,
65,406,3336,43.670921,60.132970
66,420,3309,51.076671,67.108517
67,423,3350,51.960751,66.527940
68,428,3411,53.940344,68.299595


In [ ]:
years_count = df_main.groupby('program_id')['year'].nunique()
valid_programs = years_count[years_count == 4].index
valid_programs
df_filtered = df_main[df_main['program_id'].isin(valid_programs)].copy()

print(f'Original programs: {df_main["program_id"].nunique()}')
print(f'Programs covering years 65-68: {len(valid_programs)}')
print(f'Total rows after filtering: {len(df_filtered)}')
df_main = df_filtered
df_main.head()

Original programs: 4859
Programs covering years 65-68: 2056
Total rows after filtering: 10384


,program_id,faculty,institution,min_pct,max_pct,criteria,applied,accepted,year
0,10010121300001A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,51.0196,85.4686,คณะวิศวกรรมศาสตร์ สาขาวิชาวิศวกรรมศาสตร์,710.0,465.0,65
1,10010121300501A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,69.6333,80.0961,NaN,306.0,50.0,65
2,10010121301101A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,45.3804,62.6137,NaN,97.0,25.0,65
3,10010121302101A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,61.2921,74.1294,NaN,67.0,10.0,65
4,10010121302501A,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,45.2471,53.8137,NaN,144.0,55.0,65


In [ ]:
df_main.sort_values(['program_id','year'],ascending=[True,True])[60:120]

,program_id,faculty,institution,min_pct,max_pct,criteria,applied,accepted,year
9579,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,71.7500,80.6875,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาสเปน,156.0,211.0,67
9580,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,70.8750,82.5000,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาฝรั่งเศส,404.0,211.0,67
9581,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,71.2500,79.8750,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเกาหลี,704.0,211.0,67
9582,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,70.8750,85.2500,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาญี่ปุ่น,772.0,211.0,67
9583,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,70.8750,81.7500,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาจีน,985.0,211.0,67
14292,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,76.6250,86.8750,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาคณิตศาสตร์ประย...,1388.0,20.0,68
14293,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,73.8125,85.6250,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาฝรั่งเศส,328.0,219.0,68
14294,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,73.9375,82.1250,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเยอรมัน,111.0,219.0,68
14295,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,73.7500,86.1250,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาญี่ปุ่น,668.0,219.0,68
14296,10010122904301A,คณะอักษรศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,73.8125,86.8125,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเกาหลี,409.0,219.0,68


In [ ]:
df_main['program_id'].nunique()

2056

In [ ]:
COLS = ['program_id','faculty','institution','min_pct','max_pct','criteria','applied','accepted','year']

df_all = df_main[COLS]
df_all = df_all[(df_all['min_pct'] > 0)].dropna(subset=['min_pct','program_id'])
df_all['applied']  = pd.to_numeric(df_all['applied'],  errors='coerce')
df_all['accepted'] = pd.to_numeric(df_all['accepted'], errors='coerce')
df_all['comp_rate'] = df_all['applied'] / df_all['accepted'].replace(0, np.nan)

print("ปีที่มีในข้อมูล:", sorted(df_all['year'].unique()))
for yr in [65, 66, 67, 68]:
    n = (df_all['year'] == yr).sum()
    print(f"  ปี {yr}: {n} แถว")

ปีที่มีในข้อมูล: [np.int64(65), np.int64(66), np.int64(67), np.int64(68)]
  ปี 65: 2588 แถว
  ปี 66: 2597 แถว
  ปี 67: 2577 แถว
  ปี 68: 2622 แถว


In [ ]:
df_agg = df_all.groupby(['program_id','year', 'criteria'], dropna=False).agg(
    faculty     = ('faculty',    'first'),
    institution = ('institution','first'),
    min_pct     = ('min_pct',    'min'),
    max_pct     = ('max_pct',    'max'),
    comp_rate   = ('comp_rate',  'mean'),
).reset_index()

if df_agg.empty:
    df_agg['min_z'] = pd.Series(dtype=float)
else:
    z_parts = []
    for yr, grp in df_agg.groupby('year'):
        mu, sd = grp['min_pct'].mean(), grp['min_pct'].std()
        if sd > 0:
            z_parts.append((grp['min_pct'] - mu) / sd)
        else:
            z_parts.append(pd.Series(0, index=grp['min_pct'].index))
    df_agg['min_z'] = pd.concat(z_parts)

print('ปีที่มีข้อมูล:', sorted(df_agg['year'].unique()))
print(f'หลักสูตรทั้งหมด: {df_agg["program_id"].nunique()}')

year_counts = df_agg.groupby('program_id')['year'].count()
print('\nการกระจายจำนวนปีต่อหลักสูตร:')
print(year_counts.value_counts().sort_index())
print('\nค่าเฉลี่ย min_pct แต่ละปี:')
print(df_agg.groupby('year')['min_pct'].mean().round(2))

ปีที่มีข้อมูล: [np.int64(65), np.int64(66), np.int64(67), np.int64(68)]
หลักสูตรทั้งหมด: 2056

การกระจายจำนวนปีต่อหลักสูตร:
year
4      1718
5        95
6        53
7        63
8        55
9        12
10        6
11        6
12       12
13        5
14        6
16        3
17        1
18        1
20        2
21        1
22        2
23        4
25        1
27        3
28        1
32        1
33        2
39        1
93        1
101       1
Name: count, dtype: int64

ค่าเฉลี่ย min_pct แต่ละปี:
year
65    42.72
66    50.12
67    51.92
68    53.86
Name: min_pct, dtype: float64


In [ ]:
year_counts[60:120]

,year
program_id,
10020104210701A,4
10020104210702A,4
10020104211201E,4
10020104212301A,4
10020104212501A,4
10020104212601A,4
10020104212701A,4
10020104212702A,4
10020104213301A,4


In [ ]:
df_agg

,program_id,year,criteria,faculty,institution,min_pct,max_pct,comp_rate,min_z
0,10010121300001A,65,คณะวิศวกรรมศาสตร์ สาขาวิชาวิศวกรรมศาสตร์,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,51.0196,85.4686,1.526882,0.434523
1,10010121300001A,66,คณะวิศวกรรมศาสตร์ สาขาวิชาวิศวกรรมศาสตร์,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,54.1500,80.8274,6.733333,0.215876
2,10010121300001A,67,NaN,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,53.6832,83.5332,2.944737,0.099734
3,10010121300001A,68,NaN,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,59.8830,82.6000,3.633333,0.369091
4,10010121300501A,65,NaN,คณะวิศวกรรมศาสตร์,จุฬาลงกรณ์มหาวิทยาลัย,69.6333,80.0961,6.120000,1.408690
...,...,...,...,...,...,...,...,...,...
9813,52212901111701A,68,โครงการบุคคลทั่วไป,คณะพยาบาลศาสตร์,สถาบันพระบรมราชชนก,53.1292,55.5272,17.714286,-0.044621
9814,52213001111701A,65,NaN,คณะพยาบาลศาสตร์,สถาบันพระบรมราชชนก,51.3500,61.7000,36.166667,0.451815
9815,52213001111701A,66,โครงการบุคคลทั่วไป,คณะพยาบาลศาสตร์,สถาบันพระบรมราชชนก,45.1790,49.6694,21.222222,-0.265066
9816,52213001111701A,67,โครงการบุคคลทั่วไป,คณะพยาบาลศาสตร์,สถาบันพระบรมราชชนก,48.4572,61.0402,36.461538,-0.196555


In [ ]:
available_lags = [f'min_lag{i+1}' for i in range(WINDOW)]

df_windows = build_window_dataset(df_agg, ALL_YEARS, window=WINDOW)

In [ ]:
mean_exam_score = df_windows['exam_score'].mean()
df_windows['exam_score'] = df_windows['exam_score'].fillna(mean_exam_score)

In [ ]:
df_windows

,program_id,criteria,test_year,target,exam_score,min_lag1,min_lag2
0,10010121300501A,NaN,67,72.2942,35.566200,69.6333,71.4166
1,10010121300501A,NaN,68,74.7276,36.036600,71.4166,72.2942
2,10010121301101A,NaN,67,49.1275,32.966300,45.3804,47.0276
3,10010121301101A,NaN,68,53.9274,34.153300,47.0276,49.1275
4,10010121302101A,NaN,67,58.5664,32.966300,61.2921,61.2109
...,...,...,...,...,...,...,...
3066,52212601111701A,โครงการบุคคลทั่วไป,68,48.9362,41.092567,42.2990,49.4692
3067,52212701111701A,โครงการบุคคลทั่วไป,68,49.7772,41.092567,47.1786,48.4868
3068,52212801111701A,โครงการบุคคลทั่วไป,68,52.2034,41.092567,50.4558,51.4658
3069,52212901111701A,โครงการบุคคลทั่วไป,68,53.1292,41.092567,45.0726,52.0262


In [ ]:
df_agg_temp = df_agg.copy()
nan_placeholder = '__NAN_CRITERIA_PLACEHOLDER__'
df_agg_temp['criteria'] = df_agg_temp['criteria'].fillna(nan_placeholder)

pivot_comp = df_agg_temp.pivot_table(index=['program_id', 'criteria'], columns='year', values='comp_rate')

def get_prev_comp(row):
    pid = row['program_id']
    crit = nan_placeholder if pd.isna(row['criteria']) else row['criteria']
    test_year = row['test_year']
    prev_year = test_year - 1

    if (pid, crit) in pivot_comp.index and prev_year in pivot_comp.columns:
        return pivot_comp.loc[(pid, crit), prev_year]
    return np.nan

df_windows_v2 = df_windows.copy()
df_windows_v2['prev_comp'] = df_windows_v2.apply(get_prev_comp, axis=1)

df_windows_v2 = df_windows_v2.dropna(subset=['prev_comp'])


In [ ]:
df_windows_v2

,program_id,criteria,test_year,target,exam_score,min_lag1,min_lag2,prev_comp
0,10010121300501A,NaN,67,72.2942,35.566200,69.6333,71.4166,21.876923
1,10010121300501A,NaN,68,74.7276,36.036600,71.4166,72.2942,8.907692
2,10010121301101A,NaN,67,49.1275,32.966300,45.3804,47.0276,9.320000
3,10010121301101A,NaN,68,53.9274,34.153300,47.0276,49.1275,6.680000
4,10010121302101A,NaN,67,58.5664,32.966300,61.2921,61.2109,35.700000
...,...,...,...,...,...,...,...,...
3066,52212601111701A,โครงการบุคคลทั่วไป,68,48.9362,41.092567,42.2990,49.4692,116.111111
3067,52212701111701A,โครงการบุคคลทั่วไป,68,49.7772,41.092567,47.1786,48.4868,196.500000
3068,52212801111701A,โครงการบุคคลทั่วไป,68,52.2034,41.092567,50.4558,51.4658,123.444444
3069,52212901111701A,โครงการบุคคลทั่วไป,68,53.1292,41.092567,45.0726,52.0262,71.733333


In [ ]:
print("ข้อมูลสำหรับ program_id '10010121300501A' ใน df_agg:")
display(df_agg[df_agg['program_id'] == '10010121300501A'][['year', 'criteria', 'min_pct', 'comp_rate']])

print("\nข้อมูลสำหรับ program_id '10010121300501A' ใน df_windows:")
display(df_windows[df_windows['program_id'] == '10010121300501A'])

ข้อมูลสำหรับ program_id '10010121300501A' ใน df_agg:


,year,criteria,min_pct,comp_rate
4,65,NaN,69.6333,6.120000
5,66,NaN,71.4166,21.876923
6,67,NaN,72.2942,8.907692
7,68,NaN,74.7276,5.750000



ข้อมูลสำหรับ program_id '10010121300501A' ใน df_windows:


,program_id,criteria,test_year,target,exam_score,min_lag1,min_lag2
0,10010121300501A,NaN,67,72.2942,35.5662,69.6333,71.4166
1,10010121300501A,NaN,68,74.7276,36.0366,71.4166,72.2942


In [ ]:
df_al = pd.read_excel('/content/data/alevel.xlsx')
df_tg = pd.read_excel('/content/data/Tgat-Tpat.xlsx')

for df in [df_al, df_tg]:
    df.rename(columns={
        'ปี': 'be_year',
        'รหัส': 'subject_code', 'รหัสวิชา': 'subject_code',
        'เฉลี่ย (Mean)': 'mean', 'เฉลี่ย': 'mean',
        'S.D.': 'sd',
        'มัธยฐาน (Median)': 'median',
        'มัธยฐาน': 'median',
        'ฐานนิยม (Mode)': 'mode',
        'ฐานนิยม': 'mode',
        'ต่ำสุด (Min)': 'min_score',
        'ต่ำสุด': 'min_score',
        'สูงสุด (Max)': 'max_score',
        'สูงสุด': 'max_score',
    }, inplace=True)

df_exam = pd.concat([df_al, df_tg], ignore_index=True)
df_exam['year']         = df_exam['be_year'].astype(int) - 2500
df_exam['subject_code'] = df_exam['subject_code'].astype(int)

CODE_MAP = {90:'tgat', 91:'tgat1', 92:'tgat2', 93:'tgat3',
            20:'tpat2', 30:'tpat3', 40:'tpat4', 50:'tpat5'}

def code_to_key(c):
    if c in CODE_MAP: return CODE_MAP[c]
    if 61 <= c <= 89: return f'a_lv_{c}'
    return None

df_exam['key'] = df_exam['subject_code'].map(code_to_key)
df_exam = df_exam.dropna(subset=['key','mean','sd'])

exam_mean = df_exam.pivot_table(index='year', columns='key', values='mean')
exam_sd   = df_exam.pivot_table(index='year', columns='key', values='sd')

exam_feature_means = df_exam.groupby('key')['mean'].mean().to_dict()

print('Exam stats years:', sorted(df_exam['year'].unique()))
print('Subjects:', sorted(df_exam['key'].unique()))

df_crit = pd.read_csv('/content/data/tcas_round3_full_data.csv')
VALID_KEYS = set(df_exam['key'].unique())

def parse_criteria(x):
    if pd.isna(x): return {}
    try: return {k: v for k, v in ast.literal_eval(str(x)).items() if k in VALID_KEYS}
    except: return {}

df_crit['criteria'] = df_crit['scores_criteria'].apply(parse_criteria)
criteria_lookup = dict(zip(df_crit['program_id'], df_crit['criteria']))
print(f'\nหลักสูตรที่มี scoring criteria: {len(criteria_lookup)}')

Exam stats years: [np.int64(66), np.int64(67), np.int64(68), np.int64(69)]
Subjects: ['a_lv_61', 'a_lv_62', 'a_lv_63', 'a_lv_64', 'a_lv_65', 'a_lv_66', 'a_lv_70', 'a_lv_81', 'a_lv_82', 'a_lv_83', 'a_lv_84', 'a_lv_85', 'a_lv_86', 'a_lv_87', 'a_lv_88', 'a_lv_89', 'tgat', 'tgat1', 'tgat2', 'tgat3', 'tpat2', 'tpat3', 'tpat4', 'tpat5']

หลักสูตรที่มี scoring criteria: 4048


In [ ]:
df_exam

,be_year,subject_code,ชื่อวิชา,คะแนนเต็ม,ผู้เข้าสอบ (N),mean,median,mode,sd,min_score,max_score,year,key
0,2569,61,Math1 คณิตศาสตร์ประยุกต์ 1,100,142854,20.443,18.000,15,10.849,0.000,100.000,69,a_lv_61
1,2569,62,Math2 คณิตศาสตร์ประยุกต์ 2,100,99029,39.022,35.000,27,18.595,0.000,100.000,69,a_lv_62
2,2569,63,Sci วิทยาศาสตร์ประยุกต์,100,53340,47.282,45.600,45.6,9.739,20.000,100.000,69,a_lv_63
3,2569,64,Phy ฟิสิกส์,100,107914,24.767,23.000,20,9.673,5.000,100.000,69,a_lv_64
4,2569,65,Chem เคมี,100,98811,22.488,20.000,17.5,12.755,2.500,100.000,69,a_lv_65
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,2566,93,TGAT3 สมรรถนะการทำงาน,100,262097,61.471,63.333,66.666,14.415,0.000,96.666,66,tgat3
92,2566,20,TPAT2 ความถนัดศิลปกรรมศาสตร์,100,11546,46.915,47.333,48,9.314,4.666,79.333,66,tpat2
93,2566,30,TPAT3 ความถนัดวิทย์-เทคโน-วิศวะ,100,105420,44.812,45.000,42.666,11.014,1.333,90.666,66,tpat3
94,2566,40,TPAT4 ความถนัดสถาปัตยกรรมศาสตร์,100,6125,54.912,55.000,56,11.124,6.000,92.000,66,tpat4


In [ ]:
df_crit

,university_name,faculty_name,program_name,program_id,receive_student_number,min_gpax,min_total_score,scores_criteria,link,criteria
0,จุฬาลงกรณ์มหาวิทยาลัย,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมศา...,10010121300001A,305,2.0,51.0,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 20, 'a_lv...",https://admission.chula.ac.th/admission_c1.php,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 20, 'a_lv..."
1,จุฬาลงกรณ์มหาวิทยาลัย,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมคอ...,10010121300501A,75,2.0,51.0,"{'tgat': 20, 'tpat3': 40, 'a_lv_61': 20, 'a_lv...",https://admission.chula.ac.th/admission_c1.php,"{'tgat': 20, 'tpat3': 40, 'a_lv_61': 20, 'a_lv..."
2,จุฬาลงกรณ์มหาวิทยาลัย,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมคอ...,10010121300599A,90,2.0,51.0,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 50}",https://admission.chula.ac.th/admission_c1.php,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 50}"
3,จุฬาลงกรณ์มหาวิทยาลัย,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมเคมี,10010121300601A,5,2.0,51.0,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 20, 'a_lv...",https://admission.chula.ac.th/admission_c1.php,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 20, 'a_lv..."
4,จุฬาลงกรณ์มหาวิทยาลัย,คณะวิศวกรรมศาสตร์,หลักสูตรวิศวกรรมศาสตรบัณฑิต สาขาวิชาวิศวกรรมนิ...,10010121301101A,15,2.0,48.0,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 20, 'a_lv...",https://admission.chula.ac.th/admission_c1.php,"{'tgat': 20, 'tpat3': 30, 'a_lv_61': 20, 'a_lv..."
...,...,...,...,...,...,...,...,...,...,...
4496,สถาบันพระบรมราชชนก,คณะพยาบาลศาสตร์,สาขาพยาบาลศาสตร์,52212601111701A,6,2.5,0.0,"{'gpax': 10, 'tgat': 40, 'a_lv_61': 10, 'a_lv_...",https://drive.google.com/drive/folders/1W2H_T5...,"{'tgat': 40, 'a_lv_61': 10, 'a_lv_64': 6, 'a_l..."
4497,สถาบันพระบรมราชชนก,คณะพยาบาลศาสตร์,สาขาพยาบาลศาสตร์,52212701111701A,1,2.5,0.0,"{'gpax': 10, 'tgat': 40, 'a_lv_61': 10, 'a_lv_...",https://drive.google.com/drive/folders/1W2H_T5...,"{'tgat': 40, 'a_lv_61': 10, 'a_lv_64': 6, 'a_l..."
4498,สถาบันพระบรมราชชนก,คณะพยาบาลศาสตร์,สาขาพยาบาลศาสตร์,52212801111701A,7,2.5,0.0,"{'gpax': 10, 'tgat': 40, 'a_lv_61': 10, 'a_lv_...",https://drive.google.com/drive/folders/1W2H_T5...,"{'tgat': 40, 'a_lv_61': 10, 'a_lv_64': 6, 'a_l..."
4499,สถาบันพระบรมราชชนก,คณะพยาบาลศาสตร์,สาขาพยาบาลศาสตร์,52212901111701A,5,2.5,0.0,"{'gpax': 10, 'tgat': 40, 'a_lv_61': 10, 'a_lv_...",https://drive.google.com/drive/folders/1W2H_T5...,"{'tgat': 40, 'a_lv_61': 10, 'a_lv_64': 6, 'a_l..."


In [ ]:
def weighted_exam_features(pid, year):

    criteria = criteria_lookup.get(pid, {})
    if not criteria:
        return np.nan

    df_year = df_exam[df_exam['year'] == year]

    wsum = 0.0
    total_w = 0.0
    has_any = False

    for exam_key, weight in criteria.items():
        stat_row = df_year[df_year['key'] == exam_key]
        if not stat_row.empty:
            m = stat_row['mean'].values[0]
            has_any = True
        else:
            m = exam_feature_means.get(exam_key, np.nan)
            if np.isnan(m):
                continue

        wsum += m * weight
        total_w += weight

    if total_w == 0 or not has_any:
        return np.nan

    return wsum / total_w

In [ ]:
def evaluate_metrics(y_true, y_pred, label=""):
    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    mae   = mean_absolute_error(y_true, y_pred)
    under = np.mean(y_pred < y_true) * 100
    if label:
        print(f"  RMSE           : {rmse:.4f}")
        print(f"  MAE            : {mae:.4f}")
        print(f"  Underestimate% : {under:.2f}%")
    return dict(mse=mse, rmse=rmse, mae=mae, under=under)

In [ ]:
df_all = df_main.copy()
df_all['applied']  = pd.to_numeric(df_all['applied'],  errors='coerce')
df_all['accepted'] = pd.to_numeric(df_all['accepted'], errors='coerce')
df_all['comp_rate'] = df_all['applied'] / df_all['accepted'].replace(0, np.nan)

df_agg = df_all.groupby(['program_id','year', 'criteria'], dropna=False).agg(
    faculty     = ('faculty',    'first'),
    institution = ('institution','first'),
    min_pct     = ('min_pct',    'min'),
    max_pct     = ('max_pct',    'max'),
    comp_rate   = ('comp_rate',  'mean'),
).reset_index()

In [ ]:
def build_window_dataset(df_agg, test_years, window=2):
    rows = []
    # Create a copy of df_agg and fill NaN in 'criteria' with a placeholder for pivoting
    df_agg_temp = df_agg.copy()
    nan_placeholder = '__NAN_CRITERIA_PLACEHOLDER__'
    df_agg_temp['criteria'] = df_agg_temp['criteria'].fillna(nan_placeholder)

    # Pivot min_pct using program_id and (modified) criteria as index
    pivot_min = df_agg_temp.pivot_table(index=['program_id', 'criteria'], columns='year', values='min_pct')

    for (pid, crit_placeholder) in pivot_min.index:
        # Convert placeholder back to NaN for logical checks and final output
        crit = np.nan if crit_placeholder == nan_placeholder else crit_placeholder

        for test_year in test_years:
            lag_years = list(range(test_year - window, test_year))

            if not all(yr in pivot_min.columns for yr in lag_years):
                continue
            if pivot_min.loc[(pid, crit_placeholder), lag_years].isna().any():
                continue

            # Get target data for the current test_year for this specific (pid, crit)
            if pd.isna(crit):
                target_row = df_agg[(df_agg['program_id'] == pid) &
                                    (df_agg['criteria'].isna()) &
                                    (df_agg['year'] == test_year)]
            else:
                target_row = df_agg[(df_agg['program_id'] == pid) &
                                    (df_agg['criteria'] == crit) &
                                    (df_agg['year'] == test_year)]

            if target_row.empty:
                continue

            assert len(target_row) == 1, f"Multiple target rows found for ({pid}, {crit}, {test_year})"

            row = {
                'program_id': pid,
                'criteria': crit,
                'test_year': test_year,
                'target': target_row['min_pct'].iloc[0],
                'exam_score': weighted_exam_features(pid, test_year)
            }
            for i, yr in enumerate(lag_years):
                row[f'min_lag{i+1}'] = pivot_min.loc[(pid, crit_placeholder), yr]
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
dup_check = (df_agg.groupby(['program_id','year'])['min_pct']
               .count()
               .reset_index()
               .rename(columns={'min_pct':'n_rows'}))

print("program_id + year ที่มีมากกว่า 1 แถว (มีหลาย criteria):")
print(dup_check[dup_check['n_rows'] > 1].shape[0], "กลุ่ม")
display(dup_check[dup_check['n_rows'] > 1].head(10))

example_pid = (dup_check[dup_check['n_rows'] > 1]
                 .iloc[0]['program_id'] if len(dup_check[dup_check['n_rows'] > 1]) > 0 else None)
if example_pid:
    display(df_agg[df_agg['program_id'] == example_pid]
              [['program_id','year','criteria','min_pct','comp_rate']])

program_id + year ที่มีมากกว่า 1 แถว (มีหลาย criteria):
871 กลุ่ม


,program_id,year,n_rows
36,10010122904301A,65,9
37,10010122904301A,66,10
38,10010122904301A,67,10
39,10010122904301A,68,10
51,10010123210402A,68,2
63,10010123210701A,68,2
71,10010123211001A,68,2
91,10010123212801A,68,2
103,10010123213201A,68,2
108,10010124903201A,65,29


,program_id,year,criteria,min_pct,comp_rate
36,10010122904301A,65,สาขาวิชาภูมิศาสตร์และภูมิสารสนเทศ,47.1250,15.350000
37,10010122904301A,65,สาขาวิชาสารสนเทศศึกษา,48.0625,27.800000
38,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาคณิตศาสตร์,57.2500,38.640000
39,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาจีน,64.8125,5.350000
40,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาญี่ปุ่น,65.0000,3.795833
41,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาบาลี,65.5625,1.679167
42,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาฝรั่งเศส,64.5625,2.341667
43,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเกาหลี,64.8750,4.300000
44,10010122904301A,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเยอรมัน,66.3125,0.670833
45,10010122904301A,66,สาขาวิชาภูมิศาสตร์และภูมิสารสนเทศ,66.1875,37.416667


In [ ]:
crit_years = (df_agg.groupby(['program_id','criteria'])['year']
                .nunique().reset_index()
                .rename(columns={'year':'n_years'}))

print("การกระจายจำนวนปีที่แต่ละ (program_id, criteria) มีข้อมูล:")
print(crit_years['n_years'].value_counts().sort_index())

การกระจายจำนวนปีที่แต่ละ (program_id, criteria) มีข้อมูล:
n_years
1    1858
2     538
3     438
4     442
Name: count, dtype: int64


In [ ]:
full_crit = crit_years[crit_years['n_years'] == 4]
print(f"(program_id, criteria) ที่มีครบ 4 ปี: {len(full_crit)}")
print(f"มาจาก program_id ที่ไม่ซ้ำ: {full_crit['program_id'].nunique()}")

print(f"\nprogram_id ทั้งหมดใน df_agg: {df_agg['program_id'].nunique()}")

print("\nตัวอย่าง criteria ที่มีแค่ 1 ปี (น่าจะชื่อเปลี่ยน):")
one_year = crit_years[crit_years['n_years'] == 1].merge(df_agg[['program_id','criteria','year']], on=['program_id','criteria'])
display(one_year[one_year['program_id'] == one_year['program_id'].iloc[0]])

(program_id, criteria) ที่มีครบ 4 ปี: 442
มาจาก program_id ที่ไม่ซ้ำ: 405

program_id ทั้งหมดใน df_agg: 2056

ตัวอย่าง criteria ที่มีแค่ 1 ปี (น่าจะชื่อเปลี่ยน):


,program_id,criteria,n_years,year
0,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาคณิตศาสตร์,1,65


In [ ]:
print(f"\nตัวอย่างข้อมูลสำหรับ program_id: {example_pid}")
display(df_agg[df_agg['program_id'] == example_pid][['year', 'criteria', 'min_pct', 'comp_rate']])

print(f"\nจำนวนปีที่แต่ละ (program_id, criteria) มีข้อมูลสำหรับ {example_pid}:")
display(crit_years[crit_years['program_id'] == example_pid])


ตัวอย่างข้อมูลสำหรับ program_id: 10010122904301A


,year,criteria,min_pct,comp_rate
36,65,สาขาวิชาภูมิศาสตร์และภูมิสารสนเทศ,47.1250,15.350000
37,65,สาขาวิชาสารสนเทศศึกษา,48.0625,27.800000
38,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาคณิตศาสตร์,57.2500,38.640000
39,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาจีน,64.8125,5.350000
40,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาญี่ปุ่น,65.0000,3.795833
41,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาบาลี,65.5625,1.679167
42,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาฝรั่งเศส,64.5625,2.341667
43,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเกาหลี,64.8750,4.300000
44,65,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเยอรมัน,66.3125,0.670833
45,66,สาขาวิชาภูมิศาสตร์และภูมิสารสนเทศ,66.1875,37.416667



จำนวนปีที่แต่ละ (program_id, criteria) มีข้อมูลสำหรับ 10010122904301A:


,program_id,criteria,n_years
1,10010122904301A,สาขาวิชาภูมิศาสตร์และภูมิสารสนเทศ,4
2,10010122904301A,สาขาวิชาสารสนเทศศึกษา,4
3,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาคณิตศาสตร์,1
4,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาคณิตศาสตร์ประย...,3
5,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาจีน,4
6,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาญี่ปุ่น,4
7,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาบาลี,4
8,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาฝรั่งเศส,4
9,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาสเปน,3
10,10010122904301A,สาขาวิชาอักษรศาสตร์ เลือกสอบวิชาภาษาเกาหลี,4


In [ ]:
print("แถวที่ criteria เป็น NaN ใน df_agg:")
print(df_agg[df_agg['criteria'].isna()].groupby('year')['program_id'].count())

print(f"\nprogram_id ที่มี criteria=NaN ครบ 4 ปี (ทุกปีเป็น NaN):")
nan_crit = (df_agg[df_agg['criteria'].isna()]
              .groupby('program_id')['year'].nunique()
              .reset_index()
              .rename(columns={'year':'n_years'}))
print(nan_crit['n_years'].value_counts().sort_index())
print(f"program ที่ NaN ครบ 4 ปี: {(nan_crit['n_years']==4).sum()}")

แถวที่ criteria เป็น NaN ใน df_agg:
year
65    996
66    935
67    937
68    934
Name: program_id, dtype: int64

program_id ที่มี criteria=NaN ครบ 4 ปี (ทุกปีเป็น NaN):
n_years
1     63
2     69
3     55
4    859
Name: count, dtype: int64
program ที่ NaN ครบ 4 ปี: 859


In [ ]:
def walk_forward_eval(df_windows, feature_cols, exp_name=""):
    """
    Walk-forward validation:
      รอบ 1: train test_year < รอบแรก, test = รอบแรก
      รอบ 2: train test_year < รอบสอง, test = รอบสอง
      ...
    """
    test_years = sorted(df_windows['test_year'].unique())
    all_metrics = []

    print(f"\n{'='*55}")
    print(f"  {exp_name}")
    print(f"  Features: {feature_cols}")
    print(f"{'='*55}")

    for i, test_yr in enumerate(test_years):
        if i == 0:
            train_df = df_windows[df_windows['test_year'] < test_yr]
        else:
            train_df = df_windows[df_windows['test_year'] < test_yr]

        test_df = df_windows[df_windows['test_year'] == test_yr]

        train_df = train_df.dropna(subset=feature_cols + ['target'])
        test_df  = test_df.dropna(subset=feature_cols + ['target'])

        if len(train_df) == 0 or len(test_df) == 0:
            print(f"\n  ปี {test_yr}: ข้อมูลไม่พอ (train={len(train_df)}, test={len(test_df)}) — ข้าม")
            continue

        X_train = train_df[feature_cols].values
        y_train = train_df['target'].values
        X_test  = test_df[feature_cols].values
        y_test  = test_df['target'].values

        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        print(f"\n  รอบ test_year={test_yr}  "
              f"(train={len(X_train)} แถว, test={len(X_test)} แถว)")
        m = evaluate_metrics(y_test, y_pred, label=True)
        m['test_year'] = test_yr
        m['model'] = model
        m['test_df'] = test_df.copy()
        m['y_pred'] = y_pred
        all_metrics.append(m)

    return all_metrics

In [ ]:
FEAT_EXP1 = sorted(available_lags)

results_exp1 = walk_forward_eval(df_windows, FEAT_EXP1,
                                  exp_name="EXP 1: คะแนนย้อนหลัง")


  EXP 1: คะแนนย้อนหลัง
  Features: ['min_lag1', 'min_lag2']

  ปี 67: ข้อมูลไม่พอ (train=0, test=1327) — ข้าม

  รอบ test_year=68  (train=1327 แถว, test=1744 แถว)
  RMSE           : 10.4288
  MAE            : 6.8798
  Underestimate% : 52.18%


In [ ]:
FEAT_EXP2 = ['min_lag1', 'min_lag2', 'exam_score']

results_exp2 = walk_forward_eval(df_windows, FEAT_EXP2,
                                  exp_name="EXP 2: คะแนนย้อนหลัง + ข้อสอบ")


  EXP 2: คะแนนย้อนหลัง + ข้อสอบ
  Features: ['min_lag1', 'min_lag2', 'exam_score']

  ปี 67: ข้อมูลไม่พอ (train=0, test=1327) — ข้าม

  รอบ test_year=68  (train=1327 แถว, test=1744 แถว)
  RMSE           : 10.4127
  MAE            : 6.8491
  Underestimate% : 49.20%


In [ ]:
FEAT_EXP3 = ['min_lag1', 'min_lag2', 'exam_score', 'prev_comp']

results_exp3 = walk_forward_eval(df_windows_v2, FEAT_EXP3,
                                 exp_name="EXP 3:คะแนนย้อนหลัง + ข้อสอบ + อัตราการแข่งขัน)")


  EXP 3:คะแนนย้อนหลัง + ข้อสอบ + อัตราการแข่งขัน)
  Features: ['min_lag1', 'min_lag2', 'exam_score', 'prev_comp']

  ปี 67: ข้อมูลไม่พอ (train=0, test=1327) — ข้าม

  รอบ test_year=68  (train=1327 แถว, test=1744 แถว)
  RMSE           : 10.4278
  MAE            : 6.8640
  Underestimate% : 50.00%


In [ ]:
summary_data = []

all_exps = [
    ("1", "คะแนนย้อนหลัง", results_exp1),
    ("2", "คะแนนย้อนหลัง + ข้อสอบ ", results_exp2),
    ("3", "คะแนนย้อนหลัง + ข้อสอบ + อัตราการแข่งขัน", results_exp3)
]

for name, desc, res in all_exps:
    if res:
        m = next((r for r in res if r['test_year'] == 68), res[-1])
        summary_data.append({
            'Experiment': name,
            'Description': desc,
            'RMSE': m['rmse'],
            'MAE': m['mae'],
            'Under-prediction %': m['under']
        })

df_summary = pd.DataFrame(summary_data)
df_summary

,Experiment,Description,RMSE,MAE,Under-prediction %
0,1,คะแนนย้อนหลัง,10.428792,6.879791,52.178899
1,2,คะแนนย้อนหลัง + ข้อสอบ,10.412721,6.849129,49.197248
2,3,คะแนนย้อนหลัง + ข้อสอบ + อัตราการแข่งขัน,10.427760,6.864018,50.000000
